# Data Cleaning Prep (EDA)
Goals: inspect available parquet files, identify missing/extreme values, and outline cleaning rules.


In [ ]:
from pathlib import Path
import polars as pl

data_dir = Path('..') / 'data'
parquet_paths = sorted(data_dir.glob('*.parquet'))
parquet_paths


[PosixPath('../data/commodities.parquet'),
 PosixPath('../data/day_ahead_prices.parquet'),
 PosixPath('../data/entsoe.parquet'),
 PosixPath('../data/netztransparenz.parquet'),
 PosixPath('../data/smard.parquet')]

In [ ]:
def summarize_counts(paths):
    rows = []
    for p in paths:
        df = pl.read_parquet(p)
        rows.append((p.name, df.height, len(df.columns)))
    return pl.DataFrame(rows, schema=['file', 'rows', 'cols'])

counts = summarize_counts(parquet_paths)
counts


/var/folders/mh/98qk5jbj4z336rv06wpgxbz40000gn/T/ipykernel_63839/4019059422.py:6: DataOrientationWarning: Row orientation inferred during DataFrame construction. Explicitly specify the orientation by passing `orient="row"` to silence this warning.
  return pl.DataFrame(rows, schema=['file', 'rows', 'cols'])


file,rows,cols
str,i64,i64
"""commodities.parquet""",1028,4
"""day_ahead_prices.parquet""",35064,13
"""entsoe.parquet""",135215,33
"""netztransparenz.parquet""",96,4
"""smard.parquet""",35041,16


In [ ]:
def top_missing(df: pl.DataFrame, n: int = 5):
    return (
        pl.DataFrame({
            'column': df.columns,
            'nulls': [df[c].null_count() for c in df.columns],
            'non_nulls': [df.height - df[c].null_count() for c in df.columns],
            'dtype': [str(df[c].dtype) for c in df.columns],
        })
        .sort('nulls', descending=True)
        .head(n)
    )

missing = {p.name: top_missing(pl.read_parquet(p)) for p in parquet_paths}
missing


{'commodities.parquet': shape: (4, 4)
 ┌─────────────────┬───────┬───────────┬─────────────────────────────────┐
 │ column          ┆ nulls ┆ non_nulls ┆ dtype                           │
 │ ---             ┆ ---   ┆ ---       ┆ ---                             │
 │ str             ┆ i64   ┆ i64       ┆ str                             │
 ╞═════════════════╪═══════╪═══════════╪═════════════════════════════════╡
 │ coal_price_api2 ┆ 27    ┆ 1001      ┆ Float64                         │
 │ gas_price_ttf   ┆ 24    ┆ 1004      ┆ Float64                         │
 │ co2_price_eua   ┆ 21    ┆ 1007      ┆ Float64                         │
 │ timestamp       ┆ 0     ┆ 1028      ┆ Datetime(time_unit='ms', time_… │
 └─────────────────┴───────┴───────────┴─────────────────────────────────┘,
 'day_ahead_prices.parquet': shape: (5, 4)
 ┌────────────────┬───────┬───────────┬─────────────────────────────────┐
 │ column         ┆ nulls ┆ non_nulls ┆ dtype                           │
 │ ---            ┆ 

In [ ]:
def numeric_quantiles(df: pl.DataFrame, q_low=0.01, q_high=0.99, limit=6):
    numeric = [c for c, t in zip(df.columns, df.dtypes) if t.is_numeric()]
    numeric = numeric[:limit]
    if not numeric:
        return None
    exprs = [pl.col(c).quantile(q_low).alias(f'{c}_p01') for c in numeric] + [
        pl.col(c).quantile(q_high).alias(f'{c}_p99') for c in numeric
    ]
    return df.select(exprs)

quantiles = {p.name: numeric_quantiles(pl.read_parquet(p)) for p in parquet_paths}
quantiles


{'commodities.parquet': shape: (1, 6)
 ┌────────────────┬────────────────┬────────────────┬───────────────┬───────────────┬───────────────┐
 │ gas_price_ttf_ ┆ coal_price_api ┆ co2_price_eua_ ┆ gas_price_ttf ┆ coal_price_ap ┆ co2_price_eua │
 │ p01            ┆ 2_p01          ┆ p01            ┆ _p99          ┆ i2_p99        ┆ _p99          │
 │ ---            ┆ ---            ┆ ---            ┆ ---           ┆ ---           ┆ ---           │
 │ f64            ┆ f64            ┆ f64            ┆ f64           ┆ f64           ┆ f64           │
 ╞════════════════╪════════════════╪════════════════╪═══════════════╪═══════════════╪═══════════════╡
 │ 24.775         ┆ 92.199997      ┆ 53.325001      ┆ 239.906998    ┆ 389.350006    ┆ 94.754997     │
 └────────────────┴────────────────┴────────────────┴───────────────┴───────────────┴───────────────┘,
 'day_ahead_prices.parquet': shape: (1, 12)
 ┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐

## Quick findings (from local run)
- entsoe.parquet: 96 rows, 33 cols. Heavy sparsity in cross-border import columns (e.g., flow_import_CH/NO2/DK2 mostly null).
- day_ahead_prices.parquet: 24 rows, 13 cols. No missing values; prices min ~15.7, max ~150 EUR/MWh across zones.
- netztransparenz.parquet: 96 rows, 4 cols. No missing; NRV ranges ~-1643 to 337 MW; imbalance prices ~-172 to 229 EUR/MWh.
- smard.parquet: 35,041 rows, 16 cols. Only a few (<=3) nulls in forecasts; wide ranges (e.g., load_actual ~31k–71k MW).

Notable issues:
- Sparse ENTSo-E flow imports; avoid aggressive imputation.
- SMARD tiny gaps are safe to interpolate.
- Outliers: imbalance prices negative/positive large swings; treat as real domain behavior, not errors.


## Cleaning plan
1. Standardize timestamps (rename timestamp_utc to timestamp; convert epoch ms if present).
2. Merge all parquet files on timestamp (full join).
3. Drop columns that are entirely null; drop duplicate timestamps (keep last).
4. For numeric columns with very small gaps (<=1% nulls and >1 non-null), forward/back fill.
5. Clip extreme numeric outliers at 1st/99th percentile per column (log for transparency).
6. Persist cleaned merged parquet (e.g., data/all_merged_clean.parquet).

See `energy_trading/features/clean_data.py` for the automated implementation.
